In [8]:
!pip install -q git+https://github.com/tensorflow/docs

  ERROR: Error [WinError 2] El sistema no puede encontrar el archivo especificado while executing command git version
ERROR: Cannot find command 'git' - do you have 'git' installed and in your PATH?


In [9]:
!pip install git --version


Defaulting to user installation because normal site-packages is not writeable


ERROR: Could not find a version that satisfies the requirement git (from versions: none)
ERROR: No matching distribution found for git


### IMPORTAR LIBRERIAS

In [14]:
import keras
from keras.datasets import cifar100
from keras import layers,ops
from keras.models import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten
from keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
#from tensorflow_docs.vis import embed
import tensorflow as tf
#importimageio
import matplotlib.pyplot as plt

### IMPLEMENCIÓN DE CNN PARA CLASIFICACIÓN DE IMAGENES

In [17]:

# Cargar el conjunto de datos CIFAR-100
(x_train, y_train), (x_test, y_test) = cifar100.load_data()
n_classes = 100

# Normalizar los datos de píxeles en el rango [0, 1]
x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255

# Convertir las etiquetas a vectores one-hot
y_train = to_categorical(y_train, n_classes)
y_test = to_categorical(y_test, n_classes)

# Configurar la aumentación de datos
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=False,
    )

datagen.fit(x_train)

# Definir el modelo de la red neuronal convolucional
model = Sequential([
    Conv2D(32, (3, 3), padding="same", activation="relu", input_shape=x_train.shape[1:]),
    Conv2D(32, (3, 3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), padding="same", activation="relu"),
    Conv2D(64, (3, 3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), padding="same", activation="relu"),
    Conv2D(128, (3, 3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(256, activation="relu"),
    Dense(n_classes, activation="softmax")
])


# Compilar el modelo
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])


# Entrenar el modelo con aumentación de datos
model.fit(datagen.flow(x_train, y_train, batch_size=64),
          steps_per_epoch=len(x_train) // 64,
          epochs=10,
          validation_data=(x_test, y_test))

# Evaluar el modelo
test_loss, test_acc = model.evaluate(x_test, y_test)
print('Test accuracy:', test_acc)

Epoch 1/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 256s 313ms/step - accuracy: 0.0730 - loss: 4.0567 - val_accuracy: 0.1443 - val_loss: 3.6201
Epoch 2/10
  1/781 ━━━━━━━━━━━━━━━━━━━━ 8:03 619ms/step - accuracy: 0.0625 - loss: 4.0236

C:\Users\Surface\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


781/781 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.0625 - loss: 4.0236 - val_accuracy: 0.1450 - val_loss: 3.6217
Epoch 3/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 261s 333ms/step - accuracy: 0.1821 - loss: 3.3805 - val_accuracy: 0.2369 - val_loss: 3.1289
Epoch 4/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.2031 - loss: 3.5988 - val_accuracy: 0.2324 - val_loss: 3.1423
Epoch 5/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 373s 477ms/step - accuracy: 0.2569 - loss: 3.0077 - val_accuracy: 0.2996 - val_loss: 2.8447
Epoch 6/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 16s 20ms/step - accuracy: 0.2344 - loss: 2.9784 - val_accuracy: 0.2960 - val_loss: 2.8548
Epoch 7/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 296s 378ms/step - accuracy: 0.3071 - loss: 2.7547 - val_accuracy: 0.3428 - val_loss: 2.5941
Epoch 8/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.3438 - loss: 2.6615 - val_accuracy: 0.3421 - val_loss: 2.6011
Epoch 9/10
781/781 ━━━━━━━━━━━━━━━━━━━━ 292s 374ms/step - accuracy: 0.3390 - loss: 2.5940 - val

### IMPLEMENTACIÓN DE UNA CGAN PARA GENERACIÓN DE IMAGENES

In [18]:
# Hiperparámetros
batch_size = 64
num_channels = 1
num_classes = 10
image_size = 28
latent_dim = 128

In [ ]:
# Importamos el dataset MNIST, que ya viene separado por X, y y train y test
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
all_clothes = np.concatenate([x_train, x_test])
all_labels = np.concatenate([y_train, y_test])

# Al igual que hemos hecho otras veces, escalamos las imágenes entre 0 y 1
all_clothes = all_clothes.astype("float32") / 255.0
all_clothes = np.reshape(all_clothes, (-1, 28, 28, 1))
all_labels = keras.utils.to_categorical(all_labels, 10)

# Creamos el dataset y separamos en batches
dataset = tf.data.Dataset.from_tensor_slices((all_clothes, all_labels))
dataset = dataset.shuffle(buffer_size=1024).batch(batch_size)

print(f"Shape of training images: {all_clothes.shape}")
print(f"Shape of training labels: {all_labels.shape}")

In [ ]:
generator_in_channels = latent_dim + num_classes
discriminator_in_channels = num_channels + num_classes

In [ ]:
# Creando el discriminador.
discriminator = keras.Sequential(
    [
        keras.Input(shape=(image_size, image_size, discriminator_in_channels)),
        layers.Conv2D(64, 5, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Conv2D(128, 5, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Flatten(),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="discriminator",
)


# Creando el generador.
generator = keras.Sequential(
    [
        keras.Input(shape=(generator_in_channels,)),
        layers.Dense(7 * 7 * 128),
        layers.LeakyReLU(0.2),
        layers.Reshape((7, 7, 128)),
        layers.Conv2DTranspose(128, 4, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(64, 4, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(num_channels, 4, padding="same", activation="sigmoid"),
    ],
    name="generator",
)


In [ ]:
class ConditionalGAN(keras.Model):
    def __init__(self, discriminator, generator, latent_dim):
        super().__init__()
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim
        self.seed_generator = keras.random.SeedGenerator(1337)
        self.gen_loss_tracker = keras.metrics.Mean(name="generator_loss")
        self.disc_loss_tracker = keras.metrics.Mean(name="discriminator_loss")

    @property
    def metrics(self):
        return [self.gen_loss_tracker, self.disc_loss_tracker]

    def compile(self, d_optimizer, g_optimizer, loss_fn):
        super().compile()
        self.d_optimizer = d_optimizer
        self.g_optimizer = g_optimizer
        self.loss_fn = loss_fn

    def train_step(self, data):
        # Unpack the data.
        real_images, one_hot_labels = data

        # Add dummy dimensions to the labels so that they can be concatenated with
        # the images. This is for the discriminator.
        image_one_hot_labels = one_hot_labels[:, :, None, None]
        image_one_hot_labels = ops.repeat(
            image_one_hot_labels, repeats=[image_size * image_size]
        )
        image_one_hot_labels = ops.reshape(
            image_one_hot_labels, (-1, image_size, image_size, num_classes)
        )

        # Sample random points in the latent space and concatenate the labels.
        # This is for the generator.
        batch_size = ops.shape(real_images)[0]
        random_latent_vectors = keras.random.normal(
            shape=(batch_size, self.latent_dim), seed=self.seed_generator
        )
        random_vector_labels = ops.concatenate(
            [random_latent_vectors, one_hot_labels], axis=1
        )

        # Decode the noise (guided by labels) to fake images.
        generated_images = self.generator(random_vector_labels)

        # Combine them with real images. Note that we are concatenating the labels
        # with these images here.
        fake_image_and_labels = ops.concatenate(
            [generated_images, image_one_hot_labels], -1
        )
        real_image_and_labels = ops.concatenate([real_images, image_one_hot_labels], -1)
        combined_images = ops.concatenate(
            [fake_image_and_labels, real_image_and_labels], axis=0
        )

        # Assemble labels discriminating real from fake images.
        labels = ops.concatenate(
            [ops.ones((batch_size, 1)), ops.zeros((batch_size, 1))], axis=0
        )

        # Train the discriminator.
        with tf.GradientTape() as tape:
            predictions = self.discriminator(combined_images)
            d_loss = self.loss_fn(labels, predictions)
        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(
            zip(grads, self.discriminator.trainable_weights)
        )

        # Sample random points in the latent space.
        random_latent_vectors = keras.random.normal(
            shape=(batch_size, self.latent_dim), seed=self.seed_generator
        )
        random_vector_labels = ops.concatenate(
            [random_latent_vectors, one_hot_labels], axis=1
        )

        # Assemble labels that say "all real images".
        misleading_labels = ops.zeros((batch_size, 1))

        # Train the generator (note that we should *not* update the weights
        # of the discriminator)!
        with tf.GradientTape() as tape:
            fake_images = self.generator(random_vector_labels)
            fake_image_and_labels = ops.concatenate(
                [fake_images, image_one_hot_labels], -1
            )
            predictions = self.discriminator(fake_image_and_labels)
            g_loss = self.loss_fn(misleading_labels, predictions)
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        # Monitor loss.
        self.gen_loss_tracker.update_state(g_loss)
        self.disc_loss_tracker.update_state(d_loss)
        return {
            "g_loss": self.gen_loss_tracker.result(),
            "d_loss": self.disc_loss_tracker.result(),
        }

In [ ]:
cond_gan = ConditionalGAN(
    discriminator=discriminator, generator=generator, latent_dim=latent_dim
)

cond_gan.compile(
    d_optimizer=keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
    g_optimizer=keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
    loss_fn=keras.losses.BinaryCrossentropy(from_logits=False),
)

cond_gan.fit(dataset, epochs=50)

In [ ]:
# Etiquetas a generar. Modificar la lista labels como queramos
labels = [0,1,2,3,2,2,4,5,6,7,8,9]
n_samples = len(labels)

# Extraemos el generador de la CGAN
trained_gen = cond_gan.generator

# Convertimos las etiquetas en labels a categóricas
labels = keras.utils.to_categorical(labels, num_classes)

# Generamos el ruido para las diferentes imágenes a generar.
noise = keras.random.normal(shape=(n_samples, latent_dim))
noise = ops.reshape(noise, (n_samples, latent_dim))

# Concatenamos el ruido y las etiquetas para tener el input entero del generador
noise_and_labels = ops.concatenate([noise, labels], 1)

# Generamos las imágenes con el input
fake_images = trained_gen.predict(noise_and_labels)

# Convertimos a las dimensiones originales (28x28, aunque podríamos modificar los valores) y los valores de los píxeles yendo de 0 a 255 en vez de de 0 a 1
fake_images *= 255.0
converted_images = fake_images.astype(np.uint8)
converted_images = ops.image.resize(converted_images, (28, 28)).numpy().astype(np.uint8)

In [ ]:
# Por último, mostramos las imágenes
for image in converted_images:
    plt.imshow(image[:,:,0], cmap='gray')
    plt.axis('off')
    plt.show()